# Task 3: Monthly Spending Prediction

- Type: Regression, target `spending_30d` (THB), zero-inflated (48% zeros) and heavy-tailed.
- Metric: NMAE = 1 / (1 + Σ|y−ŷ| / Σy). Maximising NMAE = minimising MAE, so the optimal prediction is the
  conditional median. We use L1 / quantile objectives only (never MSE).
- Final model:
  1. fill missing values from two exact identities between columns;
  2. two-stage conditional median: stage 1 = P(spend > 0), stage 2 = quantiles of the spending ratio to the player's
     own history among spenders; the median of the zero-inflated mixture is read off per row;
  3. 3 LightGBM seeds averaged; predictions capped at the training maximum (300,000 THB).

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from common import metrics, paths, utils

TASK = 3
utils.seed_everything()
DATA = paths.task_data(TASK)
OUT = paths.outputs_dir(TASK)

## 1. Data card & load

In [2]:
print(paths.datacard(TASK)[:500])
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
y = train["spending_30d"].values.astype(float)
FEATURES = [c for c in test.columns if c != "player_id"]
print("train", train.shape, "test", test.shape, "| zero share", (y == 0).mean().round(4),
      "| median", np.median(y), "| mean", y.mean().round(1))

TASK 3: MONTHLY SPENDING PREDICTION

TARGET VARIABLE: spending_30d (Continuous)
Predicted spending amount in THB for next 30 days

FEATURES (32 features):

1. friend_count (float)
   Number of in-game friends

2. social_interactions (float)
   Weekly gifts, trades, team activities

3. guild_membership (float)
   Guild[1]/clan membership status

4. event_participation_rate (float)
   Percentage of limited-time events
train (104000, 34) test (25889, 33) | zero share 0.4812 | median 79.435 | mean 10497.0


## 2. EDA (details in `../experiments/task3/experiments.md`)
- 48% of players spend nothing; among spenders the median is about 2,300 THB but the mean of all rows is 10,497 THB.
- `segment` (the task-2 label, given here as a feature) is the strongest signal of *whether* a player spends.
- `prev_month_spending` is the strongest signal of *how much*: corr(log prev, log y) = 0.98 among spenders.
- Each column is about 6% missing, at random (the zero share is the same with and without a value).

In [3]:
seg = train.groupby("segment")["spending_30d"].agg(n="size", p_spend=lambda s: (s > 0).mean(), median="median", mean="mean")
sp = train[(y > 0) & train["prev_month_spending"].notna()]
print("corr(log prev_month_spending, log spending_30d) among spenders:",
      round(np.corrcoef(np.log1p(sp["prev_month_spending"]), np.log1p(sp["spending_30d"]))[0, 1], 3))
seg.round(2)

corr(log prev_month_spending, log spending_30d) among spenders: 0.982


,n,p_spend,median,mean
segment,,,,
0.0,39241,0.28,0.00,73.96
1.0,24483,0.52,264.25,944.37
2.0,19762,0.69,1837.64,2283.85
3.0,14579,0.93,54760.28,65357.70


## 3. Preprocessing

In [4]:
H, TT, AVG, POD, DR = ("historical_spending", "total_transactions", "avg_transaction_value", "purchases_on_discount",
                       "discount_rate_used")


def fill_identities(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    for _ in range(3):  # a value filled by one identity can unlock the other
        m = d[TT].isna() & d[POD].notna() & d[DR].notna() & (d[DR] > 0)
        d.loc[m, TT] = np.round(d.loc[m, POD] / d.loc[m, DR])
        m = d[TT].isna() & d[H].notna() & d[AVG].notna() & (d[AVG] > 0)
        r = np.round(d.loc[m, H] / d.loc[m, AVG])
        d.loc[m[m].index[r.values > 1], TT] = r[r.values > 1]  # a ratio of 1 means tt = 0 or 1: left missing
        tt1 = d[TT].clip(lower=1)
        m = d[H].isna() & d[TT].notna() & d[AVG].notna()
        d.loc[m, H] = tt1[m] * d.loc[m, AVG]
        m = d[AVG].isna() & d[TT].notna() & d[H].notna()
        d.loc[m, AVG] = d.loc[m, H] / tt1[m]
        m = d[POD].isna() & d[TT].notna() & d[DR].notna()
        d.loc[m, POD] = d.loc[m, TT] * d.loc[m, DR]
        m = d[DR].isna() & d[TT].notna() & d[POD].notna() & (d[TT] > 0)
        d.loc[m, DR] = d.loc[m, POD] / d.loc[m, TT]
    return d


ftr, fte = fill_identities(train), fill_identities(test)
COLS = [H, TT, AVG, POD, DR]
pd.DataFrame({"train before": train[COLS].isna().sum(), "train after": ftr[COLS].isna().sum(),
              "test before": test[COLS].isna().sum(), "test after": fte[COLS].isna().sum()})

,train before,train after,test before,test after
historical_spending,6035,393,1506,95
total_transactions,6067,204,1465,49
avg_transaction_value,5991,410,1569,98
purchases_on_discount,6083,370,1505,98
discount_rate_used,6103,433,1552,110


## 4. Folds and a baseline

In [ ]:
bins = np.zeros(len(y), dtype=int)
bins[y > 0] = 1 + pd.qcut(y[y > 0], 5, labels=False)
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=utils.SEED).split(train, bins))

BASE = dict(learning_rate=0.05, num_leaves=63, min_child_samples=50, feature_fraction=0.8, bagging_fraction=0.8,
            bagging_freq=1, lambda_l2=1.0, seed=utils.SEED, deterministic=True, force_row_wise=True, n_jobs=-1, verbose=-1)


def fit_predict(params, Xtr, ytr, Xva, yva, Xte):
    dtr = lgb.Dataset(Xtr, ytr)
    m = lgb.train(params, dtr, 5000, valid_sets=[lgb.Dataset(Xva, yva, reference=dtr)],
                  callbacks=[lgb.early_stopping(200, verbose=False)])
    return m.predict(Xva, num_iteration=m.best_iteration), m.predict(Xte, num_iteration=m.best_iteration)


def cv_report(name, oof):
    f = np.array([metrics.nmae(y[va], oof[va]) for _, va in FOLDS])
    print(f"{name:32s} NMAE {metrics.nmae(y, oof):.5f} +- {f.std():.4f}  folds {np.round(f, 4).tolist()}")
    return metrics.nmae(y, oof), f.std()


t0 = time.time()
oof_base = np.zeros(len(y))
for tr, va in FOLDS:
    a, _ = fit_predict({**BASE, "objective": "l1"}, train[FEATURES].iloc[tr], np.log1p(y[tr]),
                       train[FEATURES].iloc[va], np.log1p(y[va]), train[FEATURES].iloc[va])
    oof_base[va] = np.clip(np.expm1(a), 0, None)
print("constant-median baseline NMAE", round(metrics.nmae(y, np.full_like(y, np.median(y))), 4))
results = {"LightGBM L1 on log1p(y), raw": cv_report("LightGBM L1 on log1p(y), raw", oof_base)}
print(f"({time.time() - t0:.0f}s)")

constant-median baseline NMAE 0.5
LightGBM L1 on log1p(y), raw     NMAE 0.77970 +- 0.0033  folds [0.7776, 0.7767, 0.786, 0.7792, 0.7791]
(20s)


## 5. Final model: two-stage conditional median on the ratio to history

In [ ]:
ALPHAS_Q = np.array([0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5])
SEEDS = [42, 7, 123]
CAP = 300_000  # training maximum of spending_30d (fixed rule, not tuned)


def mixture_median(p, Q):
    Q = np.sort(Q, axis=1)  # fix quantile crossing
    q = np.clip((p - 0.5) / np.maximum(p, 1e-9), 0, ALPHAS_Q[-1])
    med = np.array([np.interp(qi, ALPHAS_Q, Qi) for qi, Qi in zip(q, Q)])
    return np.where(p <= 0.5, 0.0, med)


def log_ref(df, ratio):
    ref = df["prev_month_spending"].to_numpy(float)
    ref = np.where(np.isnan(ref), df["historical_spending"].to_numpy(float) * ratio, ref)
    return np.log1p(np.nan_to_num(ref, nan=0.0))


def twostage_ratio(X, X_test, lr, lr_test, seed):
    params = {**BASE, "seed": seed}
    nz = y > 0
    t = np.log1p(y) - lr
    oof_p, test_p = np.zeros(len(X)), np.zeros(len(X_test))
    oof_Q, test_Q = np.zeros((len(X), len(ALPHAS_Q))), np.zeros((len(X_test), len(ALPHAS_Q)))
    for tr, va in FOLDS:
        a, b = fit_predict({**params, "objective": "binary"}, X.iloc[tr], nz[tr].astype(int), X.iloc[va],
                           nz[va].astype(int), X_test)
        oof_p[va], test_p = a, test_p + b / len(FOLDS)
        trp, vap = tr[nz[tr]], va[nz[va]]  # spenders only
        for j, al in enumerate(ALPHAS_Q):
            _, b = fit_predict({**params, "objective": "quantile", "alpha": al}, X.iloc[trp], t[trp],
                               X.iloc[vap], t[vap], pd.concat([X.iloc[va], X_test]))
            oof_Q[va, j] = np.expm1(b[:len(va)] + lr[va])
            test_Q[:, j] += np.expm1(b[len(va):] + lr_test) / len(FOLDS)
    return oof_p, np.clip(oof_Q, 0, None), test_p, np.clip(test_Q, 0, None)


ratio = float((ftr["prev_month_spending"] / ftr["historical_spending"]).median())
lr, lr_test = log_ref(ftr, ratio), log_ref(fte, ratio)
print(f"median prev / historical = {ratio:.4f}; rows with no reference: train {(lr == 0).sum()}, test {(lr_test == 0).sum()}")
parts = []
for s in SEEDS:
    t0 = time.time()
    parts.append(twostage_ratio(ftr[FEATURES], fte[FEATURES], lr, lr_test, s))
    cv_report(f"seed {s}", mixture_median(parts[-1][0], parts[-1][1]))
    print(f"  ({time.time() - t0:.0f}s)", flush=True)
p = np.mean([x[0] for x in parts], 0)
Q = np.expm1(np.mean([np.log1p(x[1]) for x in parts], 0))
tp = np.mean([x[2] for x in parts], 0)
TQ = np.expm1(np.mean([np.log1p(x[3]) for x in parts], 0))
oof_final = np.minimum(mixture_median(p, Q), CAP)
test_pred = np.minimum(mixture_median(tp, TQ), CAP)
results["final: two-stage ratio, 3 seeds, cap"] = cv_report("final: two-stage ratio, 3 seeds, cap", oof_final)
utils.save_preds(TASK, "final", oof_final, test_pred)
summary = pd.DataFrame(results, index=["cv_nmae", "fold_std"]).T
summary.to_csv(OUT / "cv_results.csv")
print(f"zero share  OOF {np.mean(oof_final == 0):.3f} | test {np.mean(test_pred == 0):.3f} | true {np.mean(y == 0):.3f}")
summary.round(5)

median prev / historical = 0.3499; rows with no reference: train 27, test 4


seed 42                          NMAE 0.79160 +- 0.0046  folds [0.7873, 0.7874, 0.7998, 0.7928, 0.7909]
  (243s)


seed 7                           NMAE 0.79160 +- 0.0047  folds [0.7876, 0.7871, 0.8001, 0.793, 0.7904]
  (244s)


seed 123                         NMAE 0.79136 +- 0.0046  folds [0.7874, 0.7871, 0.7996, 0.7929, 0.7901]
  (242s)


final: two-stage ratio, 3 seeds, cap NMAE 0.79188 +- 0.0046  folds [0.7877, 0.7877, 0.8001, 0.7934, 0.7908]
zero share  OOF 0.504 | test 0.504 | true 0.481


,cv_nmae,fold_std
"LightGBM L1 on log1p(y), raw",0.77970,0.00329
"final: two-stage ratio, 3 seeds, cap",0.79188,0.00459


## 6. Write output

In [7]:
final = utils.write_final_pred(TASK, test["player_id"], np.round(test_pred, 2))
final.head()

,id,prediction
0,P030374,0.00
1,P113216,2221.67
2,P015351,0.00
3,P095374,0.00
4,P059369,0.00
